<a href="https://colab.research.google.com/github/alexandertaoadams/AlexanderAdamsMastersThesis/blob/main/Benchmark2_Sony1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install gpjax
!pip install sktime
!pip install xgboost

In [ ]:
# jax libraries
import numpy as np
import jax
import jax.numpy as jnp

# gpjax libraries
import gpjax as gpx

# core libraries
from flax import nnx
import optax as ox

# metrics
from sklearn.metrics import f1_score, matthews_corrcoef

# models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

# dataset libraries
import pandas as pd
from sktime.datasets import load_from_tsfile


In [ ]:
!git clone https://github.com/alexandertaoadams/AlexanderAdamsMastersThesis.git

import sys
sys.path.insert(0, '/content/AlexanderAdamsMastersThesis')
import AlexanderAdamsMastersThesis.src as src

from src.data_manipulation import normalise, add_time
from src.inducing_variables import initial_inducing_variables
from src.datasets import Dataset_3D
from src.fit import timed_fit
from src.predict import batched_predict

from benchmarks.trivial import TrivialSignatureKernel
from src.kernels import SignatureKernel


fatal: destination path 'AlexanderAdamsMastersThesis' already exists and is not an empty directory.


# Data Preprocess

In [ ]:
file_path_train = "/content/drive/MyDrive/Dissertation/DATA_Unshared/SonyAIBORobotSurface1_TRAIN.ts"
file_path_test = "/content/drive/MyDrive/Dissertation/DATA_Unshared/SonyAIBORobotSurface1_TEST.ts"

train_data, train_labels = load_from_tsfile(file_path_train)
test_data, test_labels = load_from_tsfile(file_path_test)

train_data_2 = jnp.array((np.stack([np.stack(row) for row in train_data.to_numpy()])))
train_labels_2 = jnp.array([int(i)-1 for i in train_labels])
xtrain, train_mean, train_std = normalise(add_time(train_data_2))
ytrain = train_labels_2

test_data_2 = jnp.array((np.stack([np.stack(row) for row in test_data.to_numpy()])))
test_labels_2 = jnp.array([int(i)-1 for i in test_labels])
xtest = ((add_time(test_data_2) - train_mean) / (train_std))
ytest = test_labels_2
n_inputs, n_dims, n_steps = xtrain.shape

# Flatten for np models
xtrain_flattened = xtrain.reshape(n_inputs, -1)
xtest_flattened = xtest.reshape(xtest.shape[0], -1)

# Convert np to jnp
xtrain_jnp = jnp.array(xtrain)
ytrain_jnp = jnp.array(ytrain)
xtest_jnp = jnp.array(xtest)
ytest_jnp = jnp.array(ytest)

# Control training parameters
num_inducing = 16
batch_size = -1
max_time = 300
optim = ox.adam(learning_rate=1e-3)

# Create train dataset, initialise inducing variables
Z = initial_inducing_variables(xtrain_jnp, ytrain_jnp, num_inducing)
D = gpx.dataset.Dataset(xtrain_jnp.reshape(n_inputs, -1), jnp.expand_dims(ytrain, axis=1))
D_3 = Dataset_3D(xtrain_jnp, jnp.expand_dims(ytrain_jnp, axis=1))

NameError: name 'load_from_tsfile' is not defined

# LR

In [ ]:
model_LR = LogisticRegression()
model_LR.fit(xtrain_flattened, ytrain)
pred_LR = model_LR.predict(xtest_flattened)

MCC_LR = matthews_corrcoef(pred_LR, ytest)
F1_LR = f1_score(pred_LR, ytest)
print(f"MCC: {MCC_LR}, F1: {F1_LR}")

MCC: 0.551537439302564, F1: 0.7548291233283804


# SVC

In [ ]:
model_SVC = SVC(C=1, kernel='rbf')
model_SVC.fit(xtrain_flattened, ytrain)
pred_SVC = model_SVC.predict(xtest_flattened)

MCC_SVC = matthews_corrcoef(pred_SVC, ytest)
F1_SVC = f1_score(pred_SVC, ytest)
print(f"MCC: {MCC_SVC}, F1: {F1_SVC}")

MCC: 0.09414962411658677, F1: 0.6056338028169014


# XGBoost

In [ ]:
model_XGB = xgb.XGBClassifier(max_depth=100, max_leaves=100, n_estimators=1000)
model_XGB.fit(xtrain_flattened, ytrain)
pred_XGB = model_XGB.predict(xtest_flattened)

MCC_XGB = matthews_corrcoef(pred_XGB, ytest)
F1_XGB = f1_score(pred_XGB, ytest)
print(f"MCC: {MCC_XGB}, F1: {F1_XGB}")

MCC: 0.5142734802280486, F1: 0.7376093294460642


# RBF-SSVGP

In [ ]:
# Initialising model
RBF_kernel = gpx.kernels.RBF()
RBF_mean_function = gpx.mean_functions.Zero()
RBF_prior = gpx.gps.Prior(mean_function=RBF_mean_function, kernel=RBF_kernel)
RBF_likelihood = gpx.likelihoods.Bernoulli(num_datapoints=n_inputs)
RBF_posterior = RBF_likelihood * RBF_prior

# Model
RBF_model = gpx.variational_families.VariationalGaussian(posterior=RBF_posterior, inducing_inputs=Z)

# Training
RBF_trained_model, _ = timed_fit(
        model=RBF_model,
        objective= lambda model, data: -gpx.objectives.elbo(model, data),
        train_data=D_3,
        optim=optim,
        max_time=max_time,
        batch_size=batch_size,
    )

# Predicting
RBF_pred = jnp.round(batched_predict(xtest, batch_size=10, model=RBF_trained_model))

# Evaluating
RBF_MCC = matthews_corrcoef(RBF_pred, ytest)
RBF_F1 = f1_score(RBF_pred, ytest)
print(f"MCC: {RBF_MCC}, F1: {RBF_F1}")

MCC: 0.4912850205228385, F1: 0.7237026647966339


# Sig-SSVGP

In [ ]:
# Initialising model
Sig_kernel = TrivialSignatureKernel(n_dims, n_steps, 3)
Sig_mean_function = gpx.mean_functions.Zero()
Sig_prior = gpx.gps.Prior(mean_function=Sig_mean_function, kernel=Sig_kernel)
Sig_likelihood = gpx.likelihoods.Bernoulli(num_datapoints=n_inputs)
Sig_posterior = Sig_likelihood * Sig_prior

# Model
Sig_model = gpx.variational_families.VariationalGaussian(posterior=Sig_posterior, inducing_inputs=Z, jitter=1e-3)

# Training
Sig_trained_model, _ = timed_fit(
        model=Sig_model,
        objective= lambda model, data: -gpx.objectives.elbo(model, data),
        train_data=D_3,
        optim=optim,
        max_time=max_time,
        check_every=10,
        batch_size=batch_size,
    )

# Predicting
Sig_pred = jnp.round(batched_predict(xtest_jnp, batch_size=10, model=Sig_trained_model))

# Evaluating
Sig_MCC = matthews_corrcoef(Sig_pred, ytest)
Sig_F1 = f1_score(Sig_pred, ytest)
print(f"MCC: {Sig_MCC}, F1: {Sig_F1}")

MCC: 0.2692966216774383, F1: 0.6436781609195402


# RBFSig-SSVGP

In [ ]:
# Initialising model
RBFSig_kernel = SignatureKernel(n_dims, n_steps, 3)
RBFSig_mean_function = gpx.mean_functions.Zero()
RBFSig_prior = gpx.gps.Prior(mean_function=RBFSig_mean_function, kernel=RBFSig_kernel)
RBFSig_likelihood = gpx.likelihoods.Bernoulli(num_datapoints=n_inputs)
RBFSig_posterior = RBFSig_likelihood * RBFSig_prior

# Model
RBFSig_model = gpx.variational_families.VariationalGaussian(posterior=RBFSig_posterior, inducing_inputs=Z)

# Training
RBFSig_trained_model, _ = timed_fit(
        model=RBFSig_model,
        objective= lambda model, data: -gpx.objectives.elbo(model, data),
        train_data=D_3,
        optim=optim,
        max_time=max_time,
        check_every=10,
        batch_size=batch_size,
    )

# Predicting
RBFSig_pred = jnp.round(batched_predict(xtest_jnp, batch_size=10, model=RBFSig_trained_model))

# Evaluating
RBFSig_MCC = matthews_corrcoef(RBFSig_pred, ytest)
RBFSig_F1 = f1_score(RBFSig_pred, ytest)
print(f"MCC: {RBFSig_MCC}, F1: {RBFSig_F1}")

MCC: 0.6714587457946051, F1: 0.8152866242038217


# Results:

In [ ]:
rows = ["LR", "SVC", "XGB", "GP-RBF", "GP-Sig","GP-RBFSig"]
columns = ["MCC", "F1"]

table = pd.DataFrame(index=rows, columns=columns)
table.loc["LR"] = [MCC_LR, F1_LR]
table.loc["SVC"] = [MCC_SVC, F1_SVC]
table.loc["XGB"] = [MCC_XGB, F1_XGB]
table.loc["GP-RBF"] = [RBF_MCC, RBF_F1]
table.loc["GP-Sig"] = [Sig_MCC, Sig_F1]
table.loc["GP-RBFSig"] = [RBFSig_MCC, RBFSig_F1]

print(table)

                MCC        F1
LR         0.551537  0.754829
SVC         0.09415  0.605634
XGB        0.514273  0.737609
GP-RBF     0.491285  0.723703
GP-Sig     0.269297  0.643678
GP-RBFSig  0.671459  0.815287
